# Safety-Incident Anomaly Pattern Detection

**INT396 · SIH26165**  
This notebook is part of a reproducible project comparing Isolation Forest and Local Outlier Factor on OSHA Severe Injury Reports.

**Interpretation rule:** model flags are statistical outliers, not verified anomalies, accident predictions, or proof of unsafe conduct.

Select the project's Python environment as the notebook kernel. The raw dataset must be available at `data/raw/January2015toNovember2025.csv`.

# PCA, Agreement and Manual Review

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
from IPython.display import display, Image
from safety_anomaly.data.io import load_incidents
from safety_anomaly.data.clean import prepare_records
from safety_anomaly.features.build import build_features
from safety_anomaly.models.experiments import run_experiment_grid
from safety_anomaly.evaluation.metrics import agreement_metrics
from safety_anomaly.evaluation.review import create_blinded_review_sheet
from safety_anomaly.visualization.plots import create_pca_projection, save_pca_figure
DATA_PATH=ROOT/'data/raw/January2015toNovember2025.csv'
df=prepare_records(load_incidents(DATA_PATH))
n=min(5000,len(df))
sample=df.sample(n=n,random_state=42).sort_index().reset_index(drop=True)
features=build_features(sample, representation='structured_text', random_state=42)
summary,predictions=run_experiment_grid(features.X,random_state=42,quick=True,lof_max_rows=10000)
if_key=next(k for k in predictions if k.startswith('iforest_n100_c0.03'))
lof_key=next(k for k in predictions if k.startswith('lof_k15_c0.03'))
a=predictions[if_key]; b=predictions[lof_key]

## 1. Model agreement

In [ ]:
metrics=agreement_metrics(a['labels'],b['labels'],a['scores'],b['scores'])
display(pd.Series(metrics,name='value').to_frame())

## 2. PCA projection

In [ ]:
coords, explained=create_pca_projection(features.X,n_components=3,random_state=42)
print('Explained variance ratio:', explained)
fig_dir=ROOT/'reports/figures'; fig_dir.mkdir(parents=True,exist_ok=True)
save_pca_figure(coords,a['labels'],fig_dir/'pca_iforest.png','PCA projection: Isolation Forest')
save_pca_figure(coords,b['labels'],fig_dir/'pca_lof.png','PCA projection: LOF')
display(Image(filename=str(fig_dir/'pca_iforest.png')))
display(Image(filename=str(fig_dir/'pca_lof.png')))

## 3. Blinded manual review worksheet

In [ ]:
review=create_blinded_review_sheet(sample,a['labels'],b['labels'],sample_per_group=25,random_state=42)
private_dir=ROOT/'artifacts/private'; private_dir.mkdir(parents=True,exist_ok=True)
review_path=private_dir/'blinded_manual_review.csv'
review.to_csv(review_path,index=False)
print('Private review worksheet:',review_path)
display(review.head())

## 4. Interpretation
Jaccard, ARI and Spearman correlation describe agreement between the selected model configurations. They cannot establish which detector is correct. Reviewers should assess anonymized cases using a predefined rubric; do not expose detector membership in the review sheet.